# 🔍 01 — Exploratory Data Analysis
**Project:** Diabetic Retinopathy Detection

**Goal:** Understand the dataset — class distribution, image quality, resolution statistics — before writing a single line of model code.

---

## 📦 1. Imports & Config

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from pathlib import Path
from tqdm.notebook import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR   = Path('../data/raw')
TRAIN_DIR  = DATA_DIR / 'train_images'
TRAIN_CSV  = DATA_DIR / 'train.csv'

# ── Style ─────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('tab10')

LABEL_MAP = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
print('Setup complete.')

## 📄 2. Load & Inspect Labels CSV

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df['label_name'] = df['diagnosis'].map(LABEL_MAP)
df['image_path'] = df['id_code'].apply(lambda x: str(TRAIN_DIR / f'{x}.png'))

print(f'Total samples : {len(df):,}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

## 📊 3. Class Distribution

In [ ]:
counts = df['diagnosis'].value_counts().sort_index()
pcts   = (counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar([LABEL_MAP[i] for i in counts.index], counts.values,
                    color=sns.color_palette('tab10', 5), edgecolor='white', linewidth=1.2)
for bar, pct in zip(bars, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{pct}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Class Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=15)

# Pie chart
axes[1].pie(counts.values, labels=[LABEL_MAP[i] for i in counts.index],
            autopct='%1.1f%%', colors=sns.color_palette('tab10', 5),
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution (Proportion)', fontsize=13, fontweight='bold')

plt.suptitle('APTOS 2019 — DR Severity Class Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClass counts:')
print(pd.DataFrame({'Grade': list(LABEL_MAP.values()), 'Count': counts.values, 'Percent (%)': pcts.values}))

## 🖼️ 4. Sample Images per Class

In [ ]:
n_per_class = 4
fig, axes = plt.subplots(5, n_per_class, figsize=(16, 20))

for grade in range(5):
    samples = df[df['diagnosis'] == grade].sample(n=n_per_class, random_state=42)
    for col, (_, row) in enumerate(samples.iterrows()):
        img = cv2.imread(row['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        axes[grade, col].imshow(img)
        axes[grade, col].axis('off')
        if col == 0:
            axes[grade, col].set_ylabel(f'Grade {grade}\n{LABEL_MAP[grade]}',
                                         fontsize=12, fontweight='bold', rotation=0,
                                         labelpad=80, va='center')

plt.suptitle('Sample Retinal Images by DR Grade', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/sample_images_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 📐 5. Image Resolution Analysis

In [ ]:
heights, widths = [], []

for path in tqdm(df['image_path'].sample(500, random_state=42), desc='Reading image sizes'):
    img = cv2.imread(path)
    if img is not None:
        h, w = img.shape[:2]
        heights.append(h)
        widths.append(w)

print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
print(f'Width  — min: {min(widths)},  max: {max(widths)},  mean: {np.mean(widths):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Height Distribution'); axes[0].set_xlabel('Pixels')
axes[1].hist(widths, bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Image Width Distribution'); axes[1].set_xlabel('Pixels')
plt.tight_layout()
plt.savefig('../results/figures/resolution_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 💡 6. Image Brightness Analysis by Class

In [ ]:
brightness_records = []

for _, row in tqdm(df.sample(300, random_state=42).iterrows(), desc='Brightness analysis', total=300):
    img = cv2.imread(row['image_path'], cv2.IMREAD_GRAYSCALE)
    if img is not None:
        brightness_records.append({'diagnosis': row['diagnosis'],
                                   'mean_brightness': img.mean(),
                                   'std_brightness':  img.std()})

bdf = pd.DataFrame(brightness_records)
bdf['label'] = bdf['diagnosis'].map(LABEL_MAP)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=bdf, x='label', y='mean_brightness', ax=axes[0], palette='tab10')
axes[0].set_title('Mean Brightness by DR Grade')
axes[0].set_xlabel('Grade'); axes[0].set_ylabel('Mean Pixel Value')

sns.boxplot(data=bdf, x='label', y='std_brightness', ax=axes[1], palette='tab10')
axes[1].set_title('Brightness Std Dev by DR Grade')
axes[1].set_xlabel('Grade'); axes[1].set_ylabel('Std Dev of Pixel Values')

plt.tight_layout()
plt.savefig('../results/figures/brightness_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ 7. EDA Summary

| Finding | Implication |
|---------|-------------|
| ~49% images are Grade 0 | Use class-weighted loss; avoid raw accuracy as metric |
| Image resolutions vary widely | Fixed resize to 224×224 required before training |
| Brightness varies by grade | Ben Graham preprocessing will normalize illumination |
| Grades 0 and 1 look visually similar | Model will struggle here — QWK penalizes by distance |

**Next step:** `02_Preprocessing.ipynb` — implement Ben Graham preprocessing and build the augmentation pipeline.